In [2]:
!pip install transformers==4.52.4  langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: langchain-core
    Found ex

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto", 
    device_map="auto"
)

def generate_text(prompt, max_new_tokens=100):
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Updated to use llm_tokenizer and llm_model
    model_inputs = llm_tokenizer([text], return_tensors="pt").to(llm_model.device)

    generated_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False 
    )
    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    return llm_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
from langchain_core.language_models.llms import LLM
from typing import Any, List, Optional

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs: Any) -> str:
        # Call your existing generate_text helper function
        response_list = generate_text(prompt, max_new_tokens=500)
        
        # --- THE CRITICAL FIX: Extract the string from the list ---
        return response_list[0]
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

# Instantiate your fixed model wrapper
llm = CustomHFLLM()

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# Chain 1: Generate app idea
idea_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Come up with a creative app idea related to {topic}. Keep it short."
)
idea_chain = LLMChain(llm=llm, prompt=idea_prompt)

# Chain 2: Extract features
feature_prompt = PromptTemplate(
    input_variables=["app_idea"],
    template="Given this app idea: {app_idea}, list its top 3 features in a bullet list."
)
feature_chain = LLMChain(llm=llm, prompt=feature_prompt)

# Chain 3: Create tagline
tagline_prompt = PromptTemplate(
    input_variables=["app_idea", "features"],
    template=(
        "Based on the app idea: {app_idea} and these features:\n{features},\n"
        "write a catchy tagline (1 sentence)."
    )
)

tagline_chain = LLMChain(llm=llm, prompt=tagline_prompt)

In [17]:
def run_all(topic: str):
    print(f"Topic: {topic}")
    
    # Run the first chain
    app_idea = idea_chain.invoke({"topic": topic})["text"]
    print(f"\nApp Idea:\n{app_idea.strip()}")
    
    # Feed output into the second chain
    features = feature_chain.invoke({"app_idea": app_idea})["text"]
    print(f"\nFeatures:\n{features.strip()}")
    
    # Feed both values into the final chain
    tagline = tagline_chain.invoke({
        "app_idea": app_idea,
        "features": features
    })["text"]
    print(f"\nTagline:\n{tagline.strip()}")

In [19]:
run_all("AI dropshipping")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Topic: AI dropshipping


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



App Idea:
"AI Fashion Designer: Create unique fashion items using AI-generated designs."


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Features:
- **AI Design Generation**: Utilizes advanced algorithms to create original and innovative fashion designs.
- **Personalization Options**: Allows users to customize the design with specific color palettes, patterns, or materials.
- **Real-time Preview**: Provides instant visual feedback of how the design will look on different body types and occasions.

Tagline:
"Transform your wardrobe with AI's creative flair!"
